In [1]:
import pandas as pd
import numpy as np

# Set the seed
# Change the seed for the different pattern generation.
# seed used for the main analysis is 42.
# seeds used for the multiple data analysis are 123,25,400,30,1,222,89,901,563,1023.
np.random.seed(42)


# 1. LOAD THE GROUND TRUTH DATA

# Manually load the different generated data sets
print("Loading ground truth dataset...")
df_ground_truth = pd.read_csv('MICE_wide_1jamnew.csv') 


# 2. ISOLATE TARGET COLUMNS


# This list comprehension grabs any column starting with 'Vol_' or 'Spd_'
sensor_cols = [col for col in df_ground_truth.columns if col.startswith(('Vol_', 'Spd_'))]


# 3. INJECT MCAR MISSINGNESS 

# Define the breaking point thresholds (like 5%, 10%, 20%, 40%). we are using only 10% for the dissertation phase.
missing_rates = [0.10]

# Dictionary to store our newly degraded datasets in memory
mcar_datasets = {}

print("Simulating MCAR connectivity drops...\n")

for rate in missing_rates:
    # Create a fresh copy of the ground truth for this specific test
    df_mcar = df_ground_truth.copy()
    
    # Generate a random matrix of floats between 0.0 and 1.0, sized perfectly to sensor data
    # Because it is MCAR, every single cell gets its own independent "coin flip"
    random_matrix = np.random.rand(len(df_mcar), len(sensor_cols))
    
    # Create a boolean mask: True if the random number is less than our target rate
    mcar_mask = random_matrix < rate
    
    # Apply the mask: wherever the mask is True, replace the data with NaN
    df_mcar.loc[:, sensor_cols] = df_mcar[sensor_cols].mask(mcar_mask)
    
    # Store the degraded dataset in our dictionary
    dataset_name = f"MCAR_{int(rate*100)}"
    mcar_datasets[dataset_name] = df_mcar
    
    # Validation & Sanity Check 
    total_missing = df_mcar[sensor_cols].isna().sum().sum()
    total_cells = len(df_mcar) * len(sensor_cols)
    actual_rate = (total_missing / total_cells) * 100
    
    print(f"Target: {rate*100:02.0f}% | Actual Missing: {actual_rate:.2f}% ({total_missing:,} cells deleted)")
    
    

print("\nMCAR dataset generation complete!")

# Save all MCAR datasets 
# change the csv file for each of the data generated.
for name, df_data in mcar_datasets.items():
    filename = f"traffic_{name.lower()}.csv"
    df_data.to_csv(filename, index=False)
    print(f"Saved {filename} to your library.")

Loading ground truth dataset...
Simulating MCAR connectivity drops...

Target: 10% | Actual Missing: 10.00% (518,338 cells deleted)

MCAR dataset generation complete!
Saved traffic_mcar_10_10.csv to your library.
